## Workflow guide
This notebook first establishes the catalog and source volume, then demonstrates one patient-file ingestion, adds audit metadata, and finally applies the same pattern to every source extract.

In [0]:
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "clinical_portfolio"
SOURCE_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/source"

print(f"Catalog: {CATALOG}")
print(f"Source volume: {SOURCE_VOLUME}")

display(dbutils.fs.ls(SOURCE_VOLUME))

Catalog: workspace
Source volume: /Volumes/workspace/clinical_portfolio/source


path,name,size,modificationTime
dbfs:/Volumes/workspace/clinical_portfolio/source/condition_mapping.csv,condition_mapping.csv,313,1790244294000
dbfs:/Volumes/workspace/clinical_portfolio/source/conditions.csv,conditions.csv,311,1790244294000
dbfs:/Volumes/workspace/clinical_portfolio/source/encounters.csv,encounters.csv,421,1790244294000
dbfs:/Volumes/workspace/clinical_portfolio/source/patients.csv,patients.csv,114,1790244294000


In [0]:
patients_raw = (
    spark.read
    .option("header", True)
    .csv(f"{SOURCE_VOLUME}/patients.csv")
)

display(patients_raw)
patients_raw.printSchema()

patient_id,birth_year,gender_source_value
P001,1978,F
P002,1986,M
P003,1994,F
P004,1969,M
P005,2001,F
P006,1982,M


root
 |-- patient_id: string (nullable = true)
 |-- birth_year: string (nullable = true)
 |-- gender_source_value: string (nullable = true)



### Inspect the raw schema
CSV ingestion keeps fields as strings by design. This allows the pipeline to preserve source fidelity in Bronze; type casting and clinical validation are performed later in Silver.

In [0]:
from pyspark.sql import functions as F

bronze_patients = (
    patients_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("patients.csv"))
    .withColumn("_batch_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss"))
)

display(bronze_patients)

patient_id,birth_year,gender_source_value,_ingested_at,_source_file,_batch_id
P001,1978,F,2026-09-24T10:17:48.307Z,patients.csv,20260924101748
P002,1986,M,2026-09-24T10:17:48.307Z,patients.csv,20260924101748
P003,1994,F,2026-09-24T10:17:48.307Z,patients.csv,20260924101748
P004,1969,M,2026-09-24T10:17:48.307Z,patients.csv,20260924101748
P005,2001,F,2026-09-24T10:17:48.307Z,patients.csv,20260924101748
P006,1982,M,2026-09-24T10:17:48.307Z,patients.csv,20260924101748


### Add audit metadata
The ingestion timestamp, source file and batch identifier make every Bronze record traceable. This is the minimum audit trail needed to investigate a downstream result.

In [0]:
(
    bronze_patients.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_patients")
)

In [0]:
display(spark.table(f"{CATALOG}.{SCHEMA}.bronze_patients"))

patient_id,birth_year,gender_source_value,_ingested_at,_source_file,_batch_id
P001,1978,F,2026-09-24T10:19:43.838Z,patients.csv,20260924101943
P002,1986,M,2026-09-24T10:19:43.838Z,patients.csv,20260924101943
P003,1994,F,2026-09-24T10:19:43.838Z,patients.csv,20260924101943
P004,1969,M,2026-09-24T10:19:43.838Z,patients.csv,20260924101943
P005,2001,F,2026-09-24T10:19:43.838Z,patients.csv,20260924101943
P006,1982,M,2026-09-24T10:19:43.838Z,patients.csv,20260924101943


### Scale the pattern to every source table
After validating the patient example, the same read-metadata-write pattern is applied to encounters, conditions and the mapping table. Re-running the notebook is safe because `overwrite` makes this portfolio load idempotent.

In [0]:
source_files = {
    "patients": "patients.csv",
    "encounters": "encounters.csv",
    "conditions": "conditions.csv",
    "condition_mapping": "condition_mapping.csv",
}

for table_name, file_name in source_files.items():
    raw_df = (
        spark.read
        .option("header", True)
        .csv(f"{SOURCE_VOLUME}/{file_name}")
    )

    bronze_df = (
        raw_df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.lit(file_name))
        .withColumn("_batch_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss"))
    )

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_{table_name}")
    )

In [0]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))

database,tableName,isTemporary
clinical_portfolio,bronze_condition_mapping,false
clinical_portfolio,bronze_conditions,false
clinical_portfolio,bronze_encounters,false
clinical_portfolio,bronze_patients,false


# Bronze ingestion

This notebook loads the synthetic clinical CSV extracts into source-aligned Delta tables. Bronze deliberately keeps raw values intact while adding ingestion time, source-file and batch metadata for traceability.